In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Statistics

Every model in this book has been judged by a number computed on held-out
data: a test accuracy, a validation loss, a score on a benchmark. We read
such a number as a property of the model. It is also a property of the
particular examples that happened to land in the test set. Suppose a
classifier answers 912 of 1,000 test questions correctly. On another 1,000
questions from the same source it would score a little higher or a little
lower, so its accuracy of 91.2% is one draw of a random number. A second
model answers 921 of the same questions. Is it the better model, or did
these questions happen to suit it?

Statistics answers such questions by asking how an estimate would vary over
the samples we did not draw. That variation has a distribution, the
*sampling distribution* of the estimate, and every tool in this section
describes it or reasons with it. Its center and its width say whether an
estimate is systematically off and how much it scatters. A hypothesis test
asks whether an observed difference fits inside the scatter that chance
alone produces. A confidence interval reports the parameter values that the
data cannot rule out. When no formula describes the sampling distribution,
the bootstrap simulates it from the data themselves.

These tools matter because most decisions in deep learning are made by
comparing such numbers: which model ships, which hyperparameter wins,
whether an idea helped. A decision based on a gap smaller than its own noise
is a decision made by chance. The previous sections described how data vary
when the distribution is known; this one turns data into estimates with
error bars. The data need not come from the world. An expectation that we
estimate by averaging samples drawn by the computer, a Monte Carlo estimate,
is a sample mean like any other, and the same theory gives its error bar
(that section). A test accuracy is a coin with one flip per question, so the
coin of the earlier sections, 9 heads in 13 flips, returns along the way as
a check.

Three results are worth remembering: the mean squared error is the squared
bias plus the variance; a $p$-value is uniform under the null hypothesis;
and a confidence interval is the set of values that a test keeps.
Consistency and the delta method can be skipped on a first reading. Every
guarantee below holds for an estimator or a test chosen before the data are
seen; the end of the section says what goes wrong otherwise.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

In [ ]:
from scipy.stats import norm

In [ ]:
rng = np.random.default_rng()

## Estimators and Their Sampling Distribution

Before we can ask whether 91.2% is trustworthy, we need a name for the rule
that produced it and a model of how the test questions arrived.

### Estimators and the Sample Proportion

An *estimator* is a rule that turns data $x_1,\ldots,x_n$ into a number
$\hat\theta_n$ meant to approximate an unknown quantity $\theta$. The sample
mean, the sample median and the fraction of correct answers are estimators.
Because the data are random, so is $\hat\theta_n$. Its distribution over
repeated samples of size $n$ is its sampling distribution.

For the test accuracy, let $x_i=1$ if the classifier answers question $i$
correctly and $x_i=0$ otherwise. If the questions are drawn independently
from a large pool on which the classifier's accuracy is $\theta$, the $x_i$
are independent Bernoulli variables with mean $\theta$. The accuracy is
unknown, so we write $\theta$ for it and $\hat\theta$ for its estimate, the
fraction of correct answers among the $n$. By linearity of expectation
$\mathbb{E}[\hat\theta]=\theta$, and the variance of a frequency is
$\textrm{Var}(\hat\theta)=\theta(1-\theta)/n$ by
the equation. The estimate is centered on the true accuracy,
and its scatter shrinks like $1/\sqrt n$. At $\theta=0.912$ and
$n=1{,}000$ the standard deviation is
$\sqrt{0.912\cdot0.088/1000}\approx0.0090$, or 0.9 percentage points. By
the central limit theorem the sampling distribution is close to Gaussian, so
about 95% of test sets give an accuracy within $\pm1.8$ points of $\theta$.

We can watch the sampling distribution directly: draw 10,000 test sets of
1,000 questions for a classifier whose accuracy is $0.912$, and compute the
accuracy on each.

In [ ]:
theta, n = 0.912, 1000
acc = rng.binomial(n, theta, 10000) / n      # accuracy on 10,000 test sets
print(f'mean {acc.mean():.4f}, std {acc.std():.4f}, '
      f'formula {np.sqrt(theta * (1 - theta) / n):.4f}')

As predicted, they center on $0.912$ with a standard deviation of $0.0090$.
These 10,000 accuracies, on test sets we did not draw, are the object that
every later tool in this section reasons about.

Now the second model. If its 921 correct answers had come from an independent
test set, the gap of $0.009$ between the two accuracies would have a standard
deviation of $\sqrt{0.0090^2+0.0085^2}\approx0.012$, where $0.0085$ is the same
formula at $0.921$. The gap is 0.7 of its own standard error, so a single
comparison of this kind cannot tell the models apart. Both models in fact
answered the same questions, and that section shows
how much sharper the comparison becomes when we use this fact.

### Bias and Variance

An estimator can go wrong in two ways: it can be off on average, or it can
scatter widely around its own average. The *bias* of $\hat\theta_n$ is the
offset of its center from the truth,

$$
\textrm{Bias}(\hat\theta_n) = \mathbb{E}[\hat\theta_n] - \theta ,
$$

where the expectation runs over the random sample. An estimator whose bias
is zero for every value of $\theta$ is *unbiased*. The *variance* measures
the scatter around the estimator's own center, and its square root is the
*standard error*,

$$
\textrm{Var}(\hat\theta_n) = \mathbb{E}\bigl[(\hat\theta_n - \mathbb{E}[\hat\theta_n])^2\bigr],
\qquad
\textrm{se}(\hat\theta_n) = \sqrt{\textrm{Var}(\hat\theta_n)} .
$$

The bias says where the estimates land on average; the standard error says how
far a single estimate typically lands from there. The sample proportion has
bias $0$ and standard error $0.009$ in the example above. Variance is measured
from the estimator's center, not from $\theta$: the rule that ignores the data
and always answers $0.5$ has variance zero, and all of its error is bias.

### The Sample Variance

The sample proportion is unbiased. A biased estimator turns up as soon as a
standard error has to be computed from data: the standard error of a mean
contains the variance $\sigma^2$ of the data, which is rarely known and must be
estimated from the same sample. The natural estimate is the *plug-in variance*,
the average squared deviation from the sample mean,

$$
s_0^2 = \frac1n\sum_{i=1}^n (x_i-\bar x)^2 .
$$

It is biased low. The deviations are measured from $\bar x$, the point that
makes the sum of squared deviations smallest for this sample, while the true
mean $\mu$ is a different point at which the same sum is larger.

The smallest case shows by how much. For $n=2$ the sample mean is the
midpoint, so both deviations are $\pm(x_1-x_2)/2$, and

$$
\sum_{i=1}^2 (x_i-\bar x)^2 = \frac{(x_1-x_2)^2}{2} .
$$

The difference $x_1-x_2$ has mean $0$ and variance $2\sigma^2$, so the right
side has expectation $\sigma^2$. Dividing by $n=2$ gives an estimator whose
average is $\sigma^2/2$, half the truth; dividing by $n-1=1$ gives $\sigma^2$
on average. The same correction works for every $n$.

**Proposition (unbiased sample variance).** *Let $x_1,\ldots,x_n$ be i.i.d.
with mean $\mu$ and finite variance $\sigma^2$, and let $n\ge2$. Then*

$$
s^2 = \frac{1}{n-1}\sum_{i=1}^n (x_i-\bar x)^2
\qquad\textrm{satisfies}\qquad
\mathbb{E}[s^2] = \sigma^2 .
$$

**Proof.** We compare the deviations from $\bar x$ with those from $\mu$.
Write $x_i-\bar x=(x_i-\mu)-(\bar x-\mu)$, expand the square, and use
$\sum_i(x_i-\mu)=n(\bar x-\mu)$ in the cross term:

$$
\sum_{i=1}^n (x_i-\bar x)^2
 = \sum_{i=1}^n (x_i-\mu)^2 - 2(\bar x-\mu)\cdot n(\bar x-\mu) + n(\bar x-\mu)^2
 = \sum_{i=1}^n (x_i-\mu)^2 - n(\bar x-\mu)^2 .
$$

The cross term is twice the last term with the opposite sign, so measuring
from $\bar x$ removes $n(\bar x-\mu)^2$ from the sum. Now take expectations.
Each $\mathbb{E}[(x_i-\mu)^2]$ equals $\sigma^2$, so the first sum has
expectation $n\sigma^2$. The sample mean has variance $\sigma^2/n$, so
$\mathbb{E}[n(\bar x-\mu)^2]=\sigma^2$. Hence the sum of squared deviations
has expectation $(n-1)\sigma^2$, and dividing by $n-1$ gives
$\mathbb{E}[s^2]=\sigma^2$. $\blacksquare$

The usual intuition counts *degrees of freedom*. The $n$ deviations
$x_i-\bar x$ sum to zero, so only $n-1$ of them can vary freely: estimating
$\bar x$ from the data used up one. The plug-in variance has bias
$\frac{n-1}{n}\sigma^2-\sigma^2=-\sigma^2/n$, which vanishes as $n$ grows.

Unbiasedness does not survive a nonlinear map. Jensen's inequality for the
concave square root gives $\mathbb{E}[s]\le\sqrt{\mathbb{E}[s^2]}=\sigma$, with
strict inequality unless $s$ is constant. For Gaussian data at $n=2$, the
sample standard deviation is $s=|x_1-x_2|/\sqrt2$, and
$\mathbb{E}[s]=\sigma\sqrt{2/\pi}\approx0.80\,\sigma$: the square root of an
unbiased variance underestimates the standard deviation by 20% on average.

Numerical libraries select the denominator with a "delta degrees of
freedom" argument: `np.var(x, ddof=0)` divides by $n$, and `ddof=1` divides by
$n-1$.

### The Bias–Variance Decomposition

To see the two kinds of error side by side we need an estimator that trades
one for the other. Take $n=30$ measurements
$x_i\sim\mathcal N(\theta,\sigma^2)$ with $\theta=1$ and $\sigma=4$, a
small sample with a lot of noise. The sample mean $\bar x$ is unbiased with
variance $\sigma^2/n=16/30\approx0.533$. The *shrunk mean* $c\,\bar x$,
with a factor $0<c<1$, pulls the sample mean toward $0$. Its bias is
$c\theta-\theta=(c-1)\theta$ and its variance is $c^2\sigma^2/n$. At
$c=0.65$ the bias is $-0.35$, and the standard error drops from $0.73$ to
$0.47$. the figure draws both sampling
distributions.

![The sampling distributions of two estimators of $\theta=1$ from 30 draws of $\mathcal N(1,16)$, over many data sets. Left: the sample mean $\bar x$, with bias 0 and standard error 0.73. Right: the shrunk mean $0.65\,\bar x$, with bias $-0.35$ and standard error 0.47. The bias is the offset of the center from $\theta$; the standard error is the width.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prob-sampling-distribution.svg)

Which of the two is better? The shrunk mean is wrong on average, yet a
single estimate from it often lands closer to $\theta$. Settling the
question needs one number that counts both kinds of error. The *mean squared
error* is that number, the average squared distance of the estimator from
the truth,

$$
\textrm{MSE}(\hat\theta_n) = \mathbb{E}\bigl[(\hat\theta_n-\theta)^2\bigr] .
$$

It splits into the two errors we already have.

**Proposition (bias–variance decomposition).** *If $\hat\theta_n$ has finite
variance, then*

$$
\textrm{MSE}(\hat\theta_n) = \textrm{Bias}(\hat\theta_n)^2 + \textrm{Var}(\hat\theta_n) .
$$

**Proof.** Let $m=\mathbb{E}[\hat\theta_n]$ be the center of the estimator,
and add and subtract it inside the square:

$$
(\hat\theta_n-\theta)^2 = (\hat\theta_n-m)^2 + 2(\hat\theta_n-m)(m-\theta) + (m-\theta)^2 .
$$

Take expectations term by term. The first term gives
$\textrm{Var}(\hat\theta_n)$, and the last is the constant
$\textrm{Bias}(\hat\theta_n)^2$. The middle term vanishes, because $m-\theta$
is a constant and $\mathbb{E}[\hat\theta_n-m]=0$ by the definition of $m$.
$\blacksquare$

The decomposition is an instance of Pythagoras' theorem. Treat random variables
as vectors with the inner product $\langle U,V\rangle=\mathbb{E}[UV]$. The
fluctuation $\hat\theta_n-m$ has mean zero, which says that it is orthogonal to
every constant $a$:
$\langle\hat\theta_n-m,\,a\rangle=a\,\mathbb{E}[\hat\theta_n-m]=0$. The error
$\hat\theta_n-\theta$ is the sum of this fluctuation and the constant
$m-\theta$, so its squared length, the MSE, is the sum of the squared lengths
of two orthogonal legs, $\textrm{se}^2$ and $\textrm{Bias}^2$. In the same
geometry, $\mathbb{E}[X]$ is the orthogonal projection of $X$ onto the
constants, the projection of that section
with an inner product built from expectations.
the figure draws the triangle.

![The bias–variance decomposition as Pythagoras' theorem, for estimators of $\theta=1$ from 30 draws of $\mathcal N(1,16)$. Each estimator is a point at horizontal distance $|\textrm{bias}|$ and height $\textrm{se}$, and its distance from the origin is $\sqrt{\textrm{MSE}}$. Shrinking $\bar x$ to $0.65\,\bar x$ adds a horizontal leg of 0.35 but shortens the vertical one from 0.73 to 0.47, so the hypotenuse falls to 0.59 and the point lies inside the dashed circle of estimators as good as $\bar x$. The sample median is unbiased but lies outside it.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prob-stat-mse-triangle.svg)

The decomposition lets us rank estimators. For data sets of 30 draws it
predicts an MSE of $0.533$ for the sample mean, all of it variance, and
$0.123+0.225=0.348$ for the shrunk mean $0.65\,\bar x$. The sample median
is unbiased here by symmetry, but we have no formula for its variance yet.
The cells below measure all three over 10,000 data sets.

In [ ]:
theta, sigma, n = 1.0, 4.0, 30
data = rng.normal(theta, sigma, (10000, n))   # 10,000 data sets of size 30
estimates = {'mean': data.mean(axis=1),
             '0.65 * mean': 0.65 * data.mean(axis=1),
             'median': np.median(data, axis=1)}

In [ ]:
predicted = {'mean': sigma**2 / n,            # bias^2 + variance, by formula
             '0.65 * mean': (0.35 * theta)**2 + 0.65**2 * sigma**2 / n,
             'median': None}                  # no formula yet
print(f'{"estimator":>12} {"bias^2":>7} {"var":>6} {"MSE":>6} {"predicted":>9}')
for name, est in estimates.items():
    bias2 = (est.mean() - theta) ** 2
    var = est.var()                           # a plain average: divides by the count
    mse = ((est - theta) ** 2).mean()
    pred = '-' if predicted[name] is None else f'{predicted[name]:.3f}'
    print(f'{name:>12} {bias2:7.3f} {var:6.3f} {mse:6.3f} {pred:>9}')

Every run gives an MSE of about $0.53$ for the sample mean and about
$0.12+0.23=0.35$ for the shrunk mean, as computed. The median has about
$0.80$: it is unbiased, but its variance is half again that of $\bar x$.
A biased estimator wins, and an unbiased estimator can still be a poor one.

### The Shrunk Mean and the Trade-off

The table raises a question: how much bias should we accept in exchange for
less variance? For the shrunk mean the answer is a short calculation. By
the equation, with the bias and variance found above,

$$
\textrm{MSE}(c) = (1-c)^2\theta^2 + c^2\frac{\sigma^2}{n} .
$$

The first term grows as we shrink harder, and the second falls. Setting the
derivative $-2(1-c)\theta^2+2c\,\sigma^2/n$ to zero gives the best factor and
its error,

$$
c^\star = \frac{\theta^2}{\theta^2+\sigma^2/n},
\qquad
\textrm{MSE}(c^\star) = \frac{\theta^2\,\sigma^2/n}{\theta^2+\sigma^2/n} < \frac{\sigma^2}{n} .
$$

The best shrinkage always beats the sample mean, and it helps most when the
noise $\sigma^2/n$ is large compared with the signal $\theta^2$. At $\theta=1$,
$\sigma=4$ and $n=30$ we get $c^\star=1/1.533\approx0.65$, the factor used
above, and the MSE falls from $0.533$ to $0.348$.
the figure plots the two terms and their sum
against $c$. Here the U shape is exact: it is the sum of a falling parabola and
a rising one.

![Squared bias and variance of the shrunk mean $c\,\bar x$ as functions of the factor $c$, for $\theta=1$, $\sigma=4$ and $n=30$. Their sum, the MSE, is smallest at $c^\star\approx0.65$, where it is 0.35 against 0.53 for the sample mean at $c=1$. The curves are exact for this estimator.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prob-bias-variance-u-curve.svg)

Unfortunately, the optimal factor $c^\star$ contains the unknown $\theta$, so
it is not an estimator we can compute. A factor fixed without knowing $\theta$
can do harm. At $\theta=10$ the same $c=0.65$ has bias $-3.5$ and MSE
$12.25+0.23\approx12.5$, more than twenty times the $0.533$ of $\bar x$.
Shrinking toward $0$ helps only when the truth is near $0$. Exercise
1 replaces $\theta$ in $c^\star$ by its
estimate, and exercise 13 shows that in three or
more dimensions a similar substitution beats the sample mean for every
$\theta$.

The same trade-off governs prediction. At a fixed input $x$, the prediction
$\hat f(x)$ of a model fitted to a random training set is an estimator of
the unknown number $f^*(x)$. Its expected squared error on a fresh label is
the MSE of this estimator plus the variance of the label noise, which no
model can remove (the equation in
that section). In many families of models, more
flexibility lowers the bias and raises the variance, and regularization
plays the role of the factor $c$ [@Geman.Bienenstock.Doursat.1992].
Unlike the MSE of the shrunk mean, the error along such a family need not
trace a U: past the point where a model fits its training data exactly, the
test error can fall again (that section).

### Consistency and Efficiency

Bias and variance describe an estimator at one sample size. A different
question is whether it gets things right eventually: with enough data, does
the estimate settle on the truth? An estimator is *consistent* if it
*converges in probability* to $\theta$,

$$
P\bigl(|\hat\theta_n-\theta|>\varepsilon\bigr)\to 0
\quad\textrm{as } n\to\infty,\ \textrm{for every } \varepsilon>0 ,
$$

written $\hat\theta_n\xrightarrow{P}\theta$: the chance of missing
$\theta$ by more than any fixed margin vanishes as the sample grows.

The decomposition gives a simple sufficient condition. Markov's inequality
the equation, applied to the nonnegative variable
$(\hat\theta_n-\theta)^2$ with the threshold $\varepsilon^2$, bounds the
chance of a large miss by the MSE,

$$
P\bigl(|\hat\theta_n-\theta|>\varepsilon\bigr)
\le \frac{\textrm{MSE}(\hat\theta_n)}{\varepsilon^2}
= \frac{\textrm{Bias}(\hat\theta_n)^2+\textrm{Var}(\hat\theta_n)}{\varepsilon^2} .
$$

If both the bias and the variance tend to zero, the estimator is therefore
consistent. Neither condition suffices alone. The first observation by
itself, $\hat\theta_n=x_1$, is unbiased at every $n$, but its variance stays
$\sigma^2$ and it never settles. The estimator $\bar x+1$ has variance
$\sigma^2/n\to0$, but it settles on the wrong value. The plug-in variance
$s_0^2$ shows the remaining case: it is biased at every $n$, but its bias
$-\sigma^2/n$ and, for data with a finite fourth moment, its variance both
vanish, so it is consistent.

For the sample mean, the condition gives the law that makes averaging work.

**Proposition (weak law of large numbers).** *If $x_1,\ldots,x_n$ are i.i.d.
with mean $\mu$ and finite variance $\sigma^2$, then the sample mean
$\bar x_n=\frac1n\sum_{i=1}^n x_i$ satisfies*

$$
P\bigl(|\bar x_n - \mu| > \varepsilon\bigr) \le \frac{\sigma^2}{n\,\varepsilon^2}
\quad\textrm{for every } \varepsilon > 0,
\qquad\textrm{so}\qquad
\bar x_n \xrightarrow{P} \mu .
$$

**Proof.** By linearity $\mathbb{E}[\bar x_n]=\mu$, so the bias is zero. The
variance of a sum of independent variables is the sum of their variances, so
$\textrm{Var}(\bar x_n)=n\sigma^2/n^2=\sigma^2/n$. The bound above with
$\textrm{MSE}=\sigma^2/n$ is the displayed inequality, and its right side
tends to zero. $\blacksquare$

Finite variance is more than the law needs: a finite mean
$\mathbb{E}|x_1|<\infty$ suffices, with a longer proof
[@Wasserman.2013]. The law is why a Monte Carlo estimate of an integral
converges, why the accuracy on a large test set estimates the true accuracy,
and why the average training loss tracks the expected loss it stands in for.
The bound is crude. For our classifier it says that an accuracy measured on
1,000 questions misses by more than 3 points with probability at most
$0.0803/(1000\cdot0.03^2)\approx0.09$, while the Gaussian approximation
puts the probability near $0.001$. Its strength is that it needs nothing
beyond a finite variance; that section
derives the sharper bounds that more assumptions allow.

Consistency does not rank two estimators that are both consistent. Among
unbiased estimators, the one with the smaller variance is called more
*efficient*: it reaches the same precision with less data. For Gaussian
data the sample median is unbiased for the mean, like $\bar x$, but for
large $n$ its variance is about $\frac{\pi}{2}\,\sigma^2/n$ (exercise
9). The median therefore needs about
$\pi/2\approx1.57$ times as many observations for the same standard error,
an efficiency of $2/\pi\approx64\%$; at $n=30$ the table above shows $0.80$
against $0.53$. The sample proportion, by contrast, has the smallest
variance that any unbiased estimator of a Bernoulli parameter can have, a
floor known as the Cramér–Rao bound (that section).

### Monte Carlo Estimation

So far the sample came from the world: test questions, measurements, random
seeds. The law of large numbers does not ask where a sample comes from, and
this gives the sample mean a second use, as a way to compute. Most
expectations in deep learning (an expected loss, a normalizing constant, the
gradient of either) are integrals with no closed form, and many run over
thousands of dimensions, where no grid is affordable. Samples we can still
draw. The idea dates from the first electronic computers
[@Metropolis.Ulam.1949]: to compute $\mathbb{E}[g(X)]$ for $X$ with
density $p$, draw independent samples $x_1,\dots,x_n$ from $p$ and average,

$$
\mathbb{E}[g(X)] \approx \frac{1}{n}\sum_{i=1}^n g(x_i).
$$

The **Monte Carlo** estimate is a Riemann sum whose slices are placed by chance,
with equal weight $1/n$: regions of high density receive more samples. By the
law of large numbers the equation the average converges to
$\mathbb{E}[g(X)]$ as $n\to\infty$, and it is unbiased. How fast it
converges is its variance. If $X_1,\dots,X_n$ are independent with density $p$
and $\sigma^2=\textrm{Var}(g(X))$ is finite, then

$$
\textrm{Var}\Bigl(\frac1n\sum_{i=1}^ng(X_i)\Bigr)=\frac{\sigma^2}{n}.
$$

There is nothing new in this formula. It is the variance $\sigma^2/n$ of a
sample mean from the proof of the law of large numbers, with $g(X_i)$ in the
place of $x_i$ (the equation in
that section). What is new is the use we make of it.
The estimate's root-mean-square error is $\sigma/\sqrt n$: a hundred times
more samples give one more decimal digit. That is slow. But the rate does
not mention the dimension of $x$, and this is the whole case for sampling. It does need
a finite variance, and that section shows what heavy
tails do to an average (see also exercise
11).

A grid behaves differently. To resolution $\epsilon$ in $d$ dimensions it needs
$N=\epsilon^{-d}$ points. The left rule of that section
has error proportional to
$\epsilon=N^{-1/d}$, and a second-order rule, such as the *midpoint rule*
(tag each slice at its center), has error proportional to
$\epsilon^2=N^{-2/d}$. Monte Carlo has $N^{-1/2}$ in every dimension. In
$d=1$ the second-order grid wins; at $d=4$ the exponents tie; at $d=10$ the
grid falls to $N^{-1/5}$. The constant $\sigma$ can grow with $d$, but the
exponent cannot (exercise 10).

A single run shows what the rate means in practice. We estimate
$\int_0^1e^{-x^2}dx=\mathbb{E}[e^{-U^2}]$ for $U$ uniform on $[0,1]$, whose
exact value $\tfrac{\sqrt\pi}2\operatorname{erf}(1)=0.746824$ follows from
the definition of $\operatorname{erf}$, and print the running average after
$n$ samples, its error, and the predicted scale $\sigma/\sqrt n$, with
$\sigma=\mathrm{sd}(e^{-U^2})\approx0.2$ estimated from the samples.

In [ ]:
exact = math.sqrt(math.pi) / 2 * math.erf(1)
g = np.exp(-rng.uniform(size=10**5)**2)             # g(U), U uniform on [0, 1]
running = np.cumsum(g) / np.arange(1, 10**5 + 1)    # average of the first n
sigma = g.std()
for n in [10, 100, 1000, 10000, 100000]:
    print(f'n={n:<6}  estimate={running[n - 1]:.5f}  '
          f'error={abs(running[n - 1] - exact):.1e}  sigma/sqrt(n)={sigma / n**0.5:.1e}')
print(f'exact (sqrt(pi)/2) erf(1) = {exact:.6f}')

The errors have the predicted size, but each row is one draw of a random
error: it rarely exceeds $\sigma/\sqrt n$ by more than a factor of two or
three, and it can fall far below it. What does the error do between the
rows? We plot it at 200 values of $n$ on
logarithmic axes, against the prediction, together with the root-mean-square
error over 200 independent runs, which is what the equation
predicts.

In [ ]:
n = np.unique(np.logspace(1, 5, 200).astype(int))
sq = np.zeros(len(n))
for _ in range(200):                                # 200 independent runs
    g_run = np.exp(-rng.uniform(size=10**5)**2)
    sq += (np.cumsum(g_run)[n - 1] / n - exact)**2
rms = np.sqrt(sq / 200)                             # root-mean-square error
ratio = rms * np.sqrt(n) / sigma
print(f'RMS error / (sigma/sqrt(n)) over all n: {ratio.min():.2f} to {ratio.max():.2f}')
d2l.plot(n, [np.abs(running[n - 1] - exact), rms, sigma / np.sqrt(n)],
         'samples n', 'error', xscale='log', yscale='log',
         legend=['one run', 'RMS over 200 runs', 'sigma / sqrt(n)'])

A single running average wanders around the prediction, because its errors
at nearby $n$ share most of their samples: it can stay on one side of the
line for a long stretch, and it dips sharply wherever the error changes sign.
The root-mean-square error over the 200 runs stays within about a fifth of
$\sigma/\sqrt n$ at every $n$, so it falls with slope $-\tfrac12$ across all
four decades.

## Hypothesis Testing

An error bar describes one estimate. Most practical questions compare two:
is a new model better than the old one? This part builds a test for that
question by simulation, then derives the same test in closed form, and asks
how often it catches a real improvement.

### Comparison of Two Models

Training is random: the initialization, the order of the data and the
dropout masks all depend on a seed, so two runs of the same model reach
different accuracies. Suppose we train model A and model B with 20 seeds
each and record their test accuracies. Seed noise moves a single run by
about one percentage point, and model B is in truth better by 0.8 points.
Looking only at the 40 numbers, can we tell? The natural summary is the gap
between the two mean accuracies, which serves as the *test statistic* of
the comparison.

A test starts from the claim that it tries to refute. The *null hypothesis*
$H_0$ states that the two lists of accuracies come from the same
distribution, so that in particular the two models are equally good on
average. The *alternative* $H_A$ states that the distributions differ. We
ask whether the observed gap would be surprising if $H_0$ were true. If it
would, we *reject* $H_0$; if not, we *fail to reject* it. Failing to reject
is weaker than accepting, much as a court finds a defendant "not guilty"
rather than "innocent."

### The Permutation Test

To judge whether a gap is surprising under $H_0$ we need its distribution
under $H_0$, the *null distribution*. A *permutation test* builds it from
the data [@Fisher.1935]. If the two lists come from the same
distribution, the labels "A" and "B" carry no information: every assignment
of the 40 numbers to two groups of 20 was equally likely to produce what we
saw. So we shuffle the labels, recompute the gap, and repeat $R$ times. The
shuffled gaps show how large a gap chance alone produces. The function below
does this for all $R$ shuffles at once and counts how many shuffled gaps are
at least as large in magnitude as the observed one.

In [ ]:
def perm_p(acc_a, acc_b, rng, R=2000):
    """Two-sided permutation p-value and the R shuffled gaps."""
    k = len(acc_a)
    observed = acc_b.mean() - acc_a.mean()
    pooled = np.tile(np.concatenate([acc_a, acc_b]), (R, 1))
    shuffled = rng.permuted(pooled, axis=1)    # R relabelings
    gaps = shuffled[:, k:].mean(1) - shuffled[:, :k].mean(1)
    p = (1 + (np.abs(gaps) >= abs(observed)).sum()) / (R + 1)
    return p, gaps

Counting the observed labeling as one of the shuffles gives the
*$p$-value*,

$$
p = \frac{1+\#\{r:\ |\textrm{gap}_r|\ge|\textrm{gap}_{\textrm{obs}}|\}}{R+1} ,
$$

the probability, computed under $H_0$, of a gap at least as extreme as the
observed one. A small $p$ says that the observed gap would be rare if the
labels did not matter. This $p$-value is *two-sided*: a gap in either direction
counts. A one-sided test that asks only whether B is better counts only the
shuffled gaps at least as large as the observed one.

Before seeing the data we fix a *significance level* $\alpha$,
conventionally $\alpha=0.05$, and we reject $H_0$ when $p\le\alpha$. Here
is one simulated experiment with the numbers above.

In [ ]:
k = 20                                       # training seeds per model
acc_a = rng.normal(0.850, 0.010, k)          # per-seed accuracy, model A
acc_b = rng.normal(0.858, 0.010, k)          # model B: a true gap of 0.008
p, gaps = perm_p(acc_a, acc_b, rng, R=10000)
observed = acc_b.mean() - acc_a.mean()
print(f'observed gap {observed:.4f}, permutation p-value {p:.3f}')

The printed gap scatters around the true $0.008$ from run to run. In most
runs $p$ falls below $0.05$ and the test rejects $H_0$, but in a sizable
minority it fails to reject, as it may have done above: with this much seed
noise, 20 seeds are not always enough. The subsections below measure how
often this happens, and show that it is a property of the experiment's
design rather than a flaw of the test.

A histogram of the shuffled gaps shows the null distribution, with the
observed gap and its mirror image.

In [ ]:
counts, edges = np.histogram(gaps, bins=50)
centers = (edges[:-1] + edges[1:]) / 2
top = counts.max()
d2l.plot([centers, [observed] * 2, [-observed] * 2],
         [counts, [0, top], [0, top]], 'gap under label shuffling', 'count',
         legend=['shuffled gaps', '±observed gap'],
         fmts=('-', 'm--', 'm--'))

The shuffled gaps form a bell around zero, although the test assumed no
shape for the data, and the observed gap usually lies in a tail. The same
recipe works for any statistic that compares two groups: a difference of
medians, of error rates or of losses. (For a difference of means, the
classical counterpart in closed form is Welch's $t$-test.) The recipe is
the one every hypothesis test follows: state $H_0$ and $H_A$, fix $\alpha$
and a sample size large enough to detect the effect that matters, and only
then collect the data, compute the $p$-value, and reject $H_0$ if
$p\le\alpha$.

### The $z$-Test

Shuffling builds the null distribution by simulation. When the data are
Gaussian with known variance, the null distribution has a closed form, and so
do the $p$-value and the power. Let $x_1,\ldots,x_n$ be i.i.d.
$\mathcal N(\mu,\sigma^2)$ with $\sigma$ known, and test $H_0\!:\mu=0$ with
the standardized sample mean

$$
T = \frac{\sqrt n\,\bar x}{\sigma} .
$$

Under $H_0$, $\bar x\sim\mathcal N(0,\sigma^2/n)$, so $T$ is exactly standard
normal. Let $\Phi$ be the standard normal c.d.f. and $z_q$ its quantile,
$\Phi(z_q)=q$, so that $z_{0.975}\approx1.96$. The two-sided $p$-value is
$2(1-\Phi(|T|))$, and $p\le\alpha$ holds exactly when
$|T|\ge z_{1-\alpha/2}$. At $\alpha=0.05$ the test rejects when the sample
mean lies more than 1.96 standard errors from $0$.
the figure shows the two rejection tails.

![The rejection region of the two-sided $z$-test at $\alpha=0.05$. Under $H_0$ the statistic is standard normal; the central region holds probability $1-\alpha=0.95$, and the two tails beyond $\pm1.96$ hold $\alpha$ together. A statistic in the tails would be rare under $H_0$, so we reject $H_0$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prob-significance.svg)

The model comparison has a two-sample version of this test. With $k$ seeds
per model and per-seed standard deviation $\sigma$, the gap
$\bar x_B-\bar x_A$ has variance $2\sigma^2/k$, and
$T=(\bar x_B-\bar x_A)/(\sigma\sqrt{2/k})$ is standard normal under $H_0$. With
$\sigma=0.01$ and $k=20$ the gap has standard error $0.0032$, so a gap of
$0.008$ lies 2.5 standard errors from zero: beyond 1.96, but not by much.

### Two Kinds of Error and Power

A test can go wrong in two ways, and a good design controls both. A *type I
error*, or false positive, rejects a true $H_0$: we declare an improvement
that is not there. A *type II error*, or false negative, fails to reject a
false $H_0$: we miss a real improvement. Their rates are

$$
\alpha = P(\textrm{reject } H_0 \mid H_0 \textrm{ true}),
\qquad
\beta = P(\textrm{fail to reject } H_0 \mid H_0 \textrm{ false}),
$$

where $\beta$ is evaluated at a given alternative, such as a true gap of 0.8
points. The *power* $1-\beta$ is the probability of detecting that effect.
the figure arranges the four outcomes.

![The $2\times 2$ decision matrix of a hypothesis test: correct decisions on the diagonal, the type I error (rate $\alpha$) and the type II error (rate $\beta$) off the diagonal, and the power $1-\beta$ in the correct-rejection cell.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prob-type-i-ii-matrix.svg)

The rule "reject when $p\le\alpha$" has a type I error rate of $\alpha$
because under $H_0$ the $p$-value is uniformly distributed. The corrections
for running many tests, below, cannot be understood without this fact.

**Proposition ($p$-values are uniform under the null).** *Let the test
statistic $T$ have a continuous c.d.f. $F$ under $H_0$, and let $p=1-F(T)$
(one-sided) or $p=2\min\{F(T),1-F(T)\}$ (two-sided). Then
$P_{H_0}(p\le\alpha)=\alpha$ for every $\alpha\in[0,1]$.*

**Proof.** First, $U=F(T)$ is uniform on $[0,1]$. For $u\in(0,1)$, let $t_u$
be the largest $t$ with $F(t)=u$; it exists because $F$ is continuous and
tends to $0$ and $1$. Since $F$ is nondecreasing, $F(T)\le u$ holds exactly
when $T\le t_u$, so $P(U\le u)=F(t_u)=u$. For the one-sided $p$-value,
$P(1-U\le\alpha)=P(U\ge1-\alpha)=\alpha$. For the two-sided one,
$2\min\{U,1-U\}\le\alpha$ means $U\le\alpha/2$ or $U\ge1-\alpha/2$, and these
two disjoint events have probability $\alpha/2$ each. $\blacksquare$

For the permutation $p$-value the argument is a counting one. Under $H_0$ the
observed labeling and the $R$ shuffles are exchangeable, so the observed gap
is equally likely to hold any of the $R+1$ ranks among the $R+1$ gaps. The
$p$-value is at most $\alpha$ only when the observed gap is among the
$\lfloor\alpha(R+1)\rfloor$ largest, so $P_{H_0}(p\le\alpha)\le\alpha$ (ties
only make $p$ larger). The $+1$ in the numerator secures this guarantee.
Uniformity is what a significance level means: a test at level $\alpha$
rejects a true null in a fraction $\alpha$ of experiments.

What replaces the uniform distribution when $H_0$ is false? We run the
two-sample $z$-test of the model comparison on 2,000 simulated experiments
with 20 seeds per model, once without a true gap and once with the gap of
$0.008$, sort the $p$-values into ten bins, and count those at or below
$0.05$.

In [ ]:
se = 0.010 * np.sqrt(2 / k)                  # se of the gap, k seeds per model
for gap in (0.0, 0.008):
    acc_a = rng.normal(0.850, 0.010, (2000, k))
    acc_b = rng.normal(0.850 + gap, 0.010, (2000, k))
    T = (acc_b.mean(axis=1) - acc_a.mean(axis=1)) / se
    p = 2 * norm.sf(np.abs(T))                # two-sided z-test p-values
    hist = np.histogram(p, bins=10, range=(0, 1))[0] / 2000
    print(f'gap {gap:.3f}:', ' '.join(f'{h:.2f}' for h in hist),
          f'  p <= 0.05: {np.mean(p <= 0.05):.2f}')

Under $H_0$ each of the ten bins holds about a tenth of the $p$-values, and
about 5% of them fall at or below $0.05$. Under the gap of $0.008$ about 80%
fall below $0.1$ and 70% to 74% at or below $0.05$; this pile-up near zero
is the power of the test, which the formula below puts at $0.72$.

The $p$-value is easy to misread [@Wasserstein.Lazar.2016]. It is the
probability of data at least this extreme, computed under $H_0$. It is not
the probability that $H_0$ is true, which would require a prior and Bayes'
rule. A large $p$-value does not confirm $H_0$ either; it means only that
this experiment did not detect an effect, possibly because it could not.

How often does the model comparison detect its true gap? For the $z$-test
the power has a closed form. If the true mean is $\mu\ne0$, then $T$ is
normal with mean $\sqrt n\,\mu/\sigma$ and variance 1. Write
$\delta=|\mu|/\sigma$ for the *standardized effect*, the effect measured in
units of the noise. The test rejects when $|T|\ge z_{1-\alpha/2}$, which
happens with probability

$$
1-\beta \approx \Phi\bigl(\delta\sqrt n - z_{1-\alpha/2}\bigr) ,
$$

where we have dropped the far tail on the wrong side, of probability
$\Phi(-\delta\sqrt n-z_{1-\alpha/2})$. The power grows with the effect and with
$\sqrt n$. For $\delta=1$ and $n=8$ it is $\Phi(2.83-1.96)\approx0.81$.

In the two-sample comparison the statistic has mean $\delta\sqrt{k/2}$ with
$\delta=0.008/0.010=0.8$, so the power with $k$ seeds per model is
$\Phi(0.8\sqrt{k/2}-1.96)$: $0.72$ with 20 seeds and $0.24$ with 5. The
formula belongs to the $z$-test, which is told $\sigma$. The permutation
test is told nothing, so we measure its power on 400 fresh experiments for
each design.

In [ ]:
def power(k, gap, rng, num_experiments=400):
    """Fraction of fresh experiments in which the permutation test rejects."""
    rejections = 0
    for _ in range(num_experiments):
        acc_a = rng.normal(0.850, 0.010, k)
        acc_b = rng.normal(0.850 + gap, 0.010, k)
        rejections += perm_p(acc_a, acc_b, rng)[0] <= 0.05
    return rejections / num_experiments

for k in (20, 5):
    print(f'{k:2d} seeds: rejects in {power(k, 0.008, rng):.2f} of runs, '
          f'formula {norm.cdf(0.8 * np.sqrt(k / 2) - 1.96):.2f}')

It rejects in about 70% of the experiments with 20 seeds and about 20% with
5; with 400 experiments, each printed rate scatters by about two points
from run to run. Both rates are a little below the formula on average: the
permutation test in effect estimates the noise from the data, and that costs
it a little power. The design we ran therefore misses a real 0.8-point
improvement about three times in ten, and with five seeds about four times
in five. When a test with power $0.2$ fails to reject, we have learned
almost nothing. A common target is power $0.8$.

### Sample Size

Power can be planned before any data are collected. Setting the power equal
to $1-\beta$ gives $\delta\sqrt n-z_{1-\alpha/2}=z_{1-\beta}$, and solving
for $n$ gives the sample size needed to detect a standardized effect $\delta$
at level $\alpha$ with power $1-\beta$:

$$
n = \frac{\bigl(z_{1-\alpha/2} + z_{1-\beta}\bigr)^{2}}{\delta^{2}} .
$$

The sample size grows like $1/\delta^2$: halving the effect we want to detect
takes four times the data. At $\alpha=0.05$ and power $0.8$, the quantiles are
$z_{0.975}\approx1.960$ and $z_{0.8}\approx0.842$, so $n\approx7.85/\delta^2$.
the figure plots the power curves.

![Power of the two-sided $z$-test at $\alpha=0.05$ as a function of the sample size $n$, one curve per standardized effect $\delta$. Small-effect curves start near $\alpha$ (dotted line), where the test rejects only by chance, and every curve climbs toward $1$. The dashed line marks the target $0.8$, reached at a sample size proportional to $1/\delta^2$: about 8 observations for $\delta=1$ and about 78,500 for $\delta=0.01$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prob-power.svg)

Comparing two groups of $k$ observations each, with common standard
deviation $\sigma$, doubles the variance of the statistic,
$\textrm{Var}(\bar x_B-\bar x_A)=2\sigma^2/k$, and so doubles the sample size
needed per group:

$$
k = \frac{2\bigl(z_{1-\alpha/2} + z_{1-\beta}\bigr)^{2}}{\delta^{2}},
\qquad \delta = \frac{|\mu_B-\mu_A|}{\sigma} .
$$

For the model comparison, $\delta=0.8$ gives $k=2\cdot7.85/0.64\approx25$
seeds per model for power $0.8$. Twenty seeds fall a little short.

### Paired Comparisons

We return to the two classifiers of the introduction, with 912 and 921 correct
answers. They answered the *same* 1,000 questions, and some questions are hard
for both. Treating the two accuracies as independent samples leaves this shared
difficulty in the noise. A *paired* comparison works with the differences per
question instead. Let $d_i\in\{-1,0,1\}$ be model B's correctness minus model
A's on question $i$. On most questions the models agree, and $d_i=0$. Suppose
they disagree on 41 questions: B alone is right on 25 and A alone on 16, a net
of $+9$, which is the gap of 0.9 points. Under the null hypothesis that the two
models are interchangeable, each disagreement is equally likely to go either
way, so the right shuffle flips the sign of each $d_i$ at random. This is the
*sign-flip permutation test*.

In [ ]:
d = np.r_[np.ones(25), -np.ones(16), np.zeros(959)]   # B only, A only, agree
nonzero = d[d != 0]                                    # agreements never change
signs = rng.choice([-1, 1], size=(100000, len(nonzero)))
flipped = (signs * nonzero).sum(axis=1) / len(d)       # gaps under H_0
p = (1 + (np.abs(flipped) >= abs(d.mean())).sum()) / (len(flipped) + 1)
print(f'gap {d.mean():.3f}, sign-flip p-value {p:.2f}')

Every run gives $p\approx0.21$. (With the exact binomial count in place of
random sign flips, this is McNemar's test.) Treated as two independent
samples, the same counts give a gap of 0.7 standard errors and
$p\approx0.47$. Pairing halves the standard error of the gap, from $0.012$
to $\sqrt{41}/1000\approx0.006$, because the 959 questions on which the
models agree no longer contribute noise. The evidence is the split of 25
against 16 among the 41 disagreements, and it is still not enough: a
nine-question lead on a thousand questions does not show that B is better.

Pairing matters most when an evaluation is planned. Consider two models with
accuracies of 90% and 91% that disagree on 4% of the questions, with B alone
right on 2.5% and A alone on 1.5%. Then $d_i$ has mean $0.01$ and variance
$\mathbb{E}[d_i^2]-0.01^2=0.04-0.0001=0.0399$, a standardized effect of
$\delta=0.01/\sqrt{0.0399}\approx0.050$. By
the equation, a paired test detects the gap with power
$0.8$ on about $7.85\cdot0.0399/0.01^2\approx3{,}100$ questions. Two
independent test sets, one per model, need about
$7.85\cdot0.1719/0.01^2\approx13{,}500$ questions each, because the unpaired
gap has variance $0.9\cdot0.1+0.91\cdot0.09=0.1719$ per pair of questions.

Pairing cuts the size of each test set by a factor of about four here, and the
number of questions that must be labeled by a factor of about nine; the saving
grows when the models agree more often. Benchmark comparisons should therefore
use one test set and be analyzed per question. The same holds for seeds: when
both models are trained with the same seeds, the per-seed differences are the
units whose signs we flip (exercise
8).

### Multiple Testing

The guarantee $P(\textrm{reject}\mid H_0)=\alpha$ holds for one test fixed in
advance, and a hyperparameter sweep runs many. Compare 20 configurations with
a baseline, none of them actually better, each at $\alpha=0.05$. If the tests
are independent, the chance that at least one of them "wins" is
$1-0.95^{20}\approx0.64$. Reporting only the winners, or trying analyses
until one clears $p\le\alpha$ (*$p$-hacking*), turns noise into apparent
findings. The simplest remedy is the *Bonferroni correction*
[@Bonferroni.1936]: test each of the $m$ hypotheses at level
$\alpha/m$. If $A_j$ is the event that test $j$ rejects its true null, the
union bound gives

$$
P\Bigl(\bigcup_{j=1}^m A_j\Bigr) \le \sum_{j=1}^m P(A_j) = m\cdot\frac{\alpha}{m} = \alpha ,
$$

so the chance of any false positive stays at most $\alpha$, with no
assumption about how the tests depend on each other. For 20 independent
tests at $0.05/20$, the chance is $1-(1-0.0025)^{20}\approx0.049$.

When $m$ runs into the hundreds, the Bonferroni threshold is so strict that
real effects are missed. Large-scale screening then controls the *false
discovery rate* instead, the expected fraction of false positives among the
rejected hypotheses. The procedure of Benjamini and Hochberg
[@Benjamini.Hochberg.1995] controls it at level $\alpha$ when the
$p$-values are independent or positively dependent; exercise
3 states the rule and tests it.

## Confidence Intervals

A $p$-value answers a yes-or-no question about one value of the parameter.
Often we want the complementary view: the whole range of values that the
data allow, reported as an error bar around the estimate.

### Definition and Interpretation

A *confidence interval* is built by a procedure that captures the fixed
unknown parameter in a stated fraction of repeated experiments
[@Neyman.1937]. Formally, a $1-\alpha$ confidence interval is an
interval $C_n$ computed from the data such that

$$
P_\theta(C_n \ni \theta) \ge 1 - \alpha \quad \textrm{for every } \theta .
$$

We write $C_n\ni\theta$ to stress where the randomness lives: $\theta$ is
fixed, and the interval changes from data set to data set. The guarantee
describes the procedure. Over many data sets, at least 95% of the 95%
intervals contain $\theta$; a given interval either contains it or does not,
and nothing in the interval tells which. the figure
shows 100 intervals from 100 independent data sets. Many intervals used in
practice meet the guarantee only in the limit: an *asymptotic* $1-\alpha$
interval satisfies $\liminf_{n\to\infty}P_\theta(C_n\ni\theta)\ge1-\alpha$
for every $\theta$.

![One hundred 95% confidence intervals, each computed from its own independent data set, against the fixed true mean (dashed line). Most intervals contain it; the orange ones miss. The 95% describes the long-run hit rate of the procedure; a single interval carries no mark of whether it contains the truth.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prob-coverage.svg)

### The Gaussian Interval

We first build an interval in the simplest case, where every step is
exact. Take $x_1,\ldots,x_n$ i.i.d. $\mathcal N(\mu,\sigma^2)$ with
$\sigma$ known. Then $\bar x\sim\mathcal N(\mu,\sigma^2/n)$, so with
$h=z_{1-\alpha/2}\,\sigma/\sqrt n$ the estimate lands within $h$ of $\mu$
with probability exactly $1-\alpha$. Now read the same event as a statement
about $\mu$:

$$
|\bar x-\mu|\le h
\quad\Longleftrightarrow\quad
\mu\in[\bar x-h,\ \bar x+h] ,
$$

so $P_\mu(\mu\in[\bar x-h,\bar x+h])=1-\alpha$ for every $\mu$. This
change of viewpoint is the whole construction. At
$\alpha=0.05$ the interval is $\bar x\pm1.96\,\sigma/\sqrt n$.

When $\sigma$ is unknown, we replace it by the sample standard deviation
$s$ of the equation. The studentized statistic
$T=(\bar x-\mu)/(s/\sqrt n)$ is then no longer normal. For Gaussian data it
has Student's $t$ distribution with $n-1$ degrees of freedom
[@Student.1908], whose tails are heavier because $s$ fluctuates too.
We take this fact on faith; exercise 7
builds the same distribution as a Gaussian with a random precision. Inverting
as before, with the quantile $t_{n-1,1-\alpha/2}$ of the $t$ distribution in
place of $z_{1-\alpha/2}$, gives the interval

$$
\left[\bar x - t_{n-1,1-\alpha/2}\,\frac{s}{\sqrt n},\; \bar x + t_{n-1,1-\alpha/2}\,\frac{s}{\sqrt n}\right] ,
$$

whose coverage is exactly $1-\alpha$ for Gaussian data at every $n\ge2$. The
quantile matters at small $n$: $t_{n-1,0.975}$ is $12.71$ at $n=2$, $2.776$
at $n=5$ and $1.984$ at $n=100$, and it approaches $1.96$ as $n$ grows. The
interval used in practice replaces the quantile by its limit,

$$
\left[\bar x - 1.96\,\frac{s}{\sqrt n},\; \bar x + 1.96\,\frac{s}{\sqrt n}\right] ,
$$

an asymptotic 95% interval that is too short at every finite $n$: for
Gaussian data its coverage is $P(|t_{n-1}|\le1.96)$, which is $0.947$ at
$n=100$ but only $0.878$ at $n=5$.

For data that are not Gaussian, the central limit theorem
(that section) states that $\sqrt n(\bar x-\mu)/\sigma$
approaches a standard normal distribution whenever the variance is finite,
and $s$ approaches $\sigma$ by the law of large numbers. So
the equation is an asymptotic 95% interval for the mean
of any distribution with finite variance. How large $n$ must be depends on
the distribution. Skewed data need more observations (exercise
4), and an accuracy near 100% needs
far more, because then many test sets report a perfect score (exercise
5). In every case the half-width
shrinks like $1/\sqrt n$, so halving an error bar takes four times the data.

An accuracy is the mean of 0/1 indicators with $s^2\approx\hat\theta(1-\hat\theta)$,
so the same interval applies to it as
$\hat\theta\pm1.96\sqrt{\hat\theta(1-\hat\theta)/n}$, often called the *Wald
interval*. We compute it for our classifier and for the chapter's coin, 9
heads in 13 flips.

In [ ]:
for n_correct, n in ((912, 1000), (9, 13)):
    theta_hat = n_correct / n
    half = 1.96 * np.sqrt(theta_hat * (1 - theta_hat) / n)
    print(f'{n_correct:3d} of {n:4d}: '
          f'({theta_hat - half:.3f}, {theta_hat + half:.3f})')

The classifier's accuracy is $91.2\%\pm1.8$ points. The coin's interval
$(0.44,0.94)$ contains $\tfrac12$, so 13 flips cannot show that the coin is
biased. Narrowing its interval to $\pm0.05$ would take about
$1.96^2\cdot0.69\cdot0.31/0.05^2\approx330$ flips.

A Bayesian *credible interval* for the same coin answers a different
question. It treats $\theta$ as random with a prior distribution, and it
reports an interval that holds posterior probability $1-\alpha$ given the
observed data. A uniform prior gives the posterior $\mathrm{Beta}(10,5)$
(that section), whose central 95% interval, between
the 2.5% and 97.5% posterior quantiles, is $(0.42,0.87)$. The two intervals
are similar here, but they make different statements. The credible interval
says that, given the prior, $\theta$ lies in it with probability 0.95. The
confidence interval says that the procedure that produced it contains the
true $\theta$ in at least 95% of experiments, whatever $\theta$ is. The
Wald interval, moreover, meets that guarantee only approximately, for large
$n$: at 13 flips its coverage is about 91% at $\theta=\frac12$ and below
50% at $\theta=0.05$.

Coverage is a statement about many data sets, so it can be measured. We
draw 1,000 data sets of $n=100$ standard normal observations, build the
interval the equation from each, and count how many
contain the true mean $0$.

In [ ]:
mu, n, trials = 0.0, 100, 1000
data = rng.normal(mu, 1.0, (trials, n))           # one row = one data set
mu_hat = data.mean(axis=1)
se = data.std(axis=1, ddof=1) / np.sqrt(n)
hits = np.abs(mu_hat - mu) <= 1.96 * se           # does the interval contain mu?
print(f'first interval ({mu_hat[0] - 1.96 * se[0]:.3f}, '
      f'{mu_hat[0] + 1.96 * se[0]:.3f}) contains 0: {hits[0]}')
print(f'{hits.sum()} of {trials} intervals contain the true mean '
      f'(nominal 950, exact 947)')

The first interval contains $0$ in about 95% of runs. The count lands near
947, typically within $\pm14$, rather than near the nominal 950. The
shortfall is systematic. At $n=100$ the exact quantile $t_{99,0.975}=1.984$ is
slightly larger than $1.96$, and the coverage of the 1.96 interval is
$P(|t_{99}|\le1.96)=0.947$. The spread of $\pm14$ is two binomial standard
deviations of the count, $2\sqrt{1000\cdot0.947\cdot0.053}\approx14$. The
same construction gives the error bar on a mean accuracy over $k$ training
seeds: the seed-to-seed standard deviation divided by $\sqrt k$, times the
appropriate quantile.

### Tests and Intervals

The number 1.96 has now appeared twice, as the rejection threshold of the
$z$-test and as the multiplier of the interval's half-width. The two uses
are one fact seen from two sides.

For Gaussian data with known $\sigma$, the $z$-test of $H_0\!:\mu=\mu_0$ keeps
$H_0$ when $|T|\le z_{1-\alpha/2}$, with $T=\sqrt n(\bar x-\mu_0)/\sigma$. That
is the condition $|\bar x-\mu_0|\le h$ with the half-width
$h=z_{1-\alpha/2}\,\sigma/\sqrt n$ from above, and it says that $\mu_0$ lies in
$[\bar x-h,\,\bar x+h]$. So *a 95% confidence interval is the set of null
values that a 5% test keeps*. Coverage follows from the level of the test: for
every $\mu$,
$P_\mu(\mu\in[\bar x-h,\bar x+h])=1-P_\mu(\textrm{the test of }H_0\!:\mu\textrm{ rejects})=1-\alpha$.

The same construction works for any family of tests. Suppose that for each
$\theta_0$ we have a level-$\alpha$ test of $H_0\!:\theta=\theta_0$. The
set of values $\theta_0$ that it does not reject contains the true $\theta$
with probability at least $1-\alpha$, by the same one-line argument, so it
is a $1-\alpha$ confidence set. This is how intervals are built when no
formula inverts the test. For 29 correct answers out of 30, the Wald
interval reaches past an accuracy of 1, while inverting an exact binomial
test gives an interval inside $[0,1]$ (exercise
5).

The duality also says what to report for a comparison: an interval for the
difference, such as $\bar x_B-\bar x_A\pm1.96\,\sigma\sqrt{2/k}$ for two
models over $k$ seeds. It excludes $0$ exactly when the test rejects, and it
shows the size of the effect as well.

### The Delta Method

We often report not $\hat\theta$ itself but a function of it: the log-odds of
an accuracy, the ratio of two losses, a relative improvement. Its error bar
follows from the error bar of $\hat\theta$. If $g$ is differentiable at
$\theta$, the first-order Taylor expansion

$$
g(\hat\theta)\approx g(\theta)+g'(\theta)\,(\hat\theta-\theta)
$$

says that near $\theta$ the map $g$ stretches deviations by the factor
$|g'(\theta)|$. The standard error therefore propagates as

$$
\textrm{se}\bigl(g(\hat\theta)\bigr) \approx \bigl|g'(\hat\theta)\bigr|\,\textrm{se}(\hat\theta) ,
$$

where the unknown $\theta$ in $g'$ has been replaced by its estimate. This
rule is the *delta method* [@Wasserman.2013]. For an accuracy
$\hat\theta=0.90$ on $n=1{,}000$ questions,
$\textrm{se}(\hat\theta)=\sqrt{0.9\cdot0.1/1000}\approx0.0095$. Its log-odds
$g(\hat\theta)=\log\frac{\hat\theta}{1-\hat\theta}\approx2.20$ has slope
$g'(\hat\theta)=1/(\hat\theta(1-\hat\theta))\approx11.1$, so the log-odds has
a standard error of about $11.1\cdot0.0095\approx0.105$.

For a vector of estimates with covariance matrix $\boldsymbol\Sigma$ the
same expansion gives
$\textrm{Var}(g(\hat{\boldsymbol\theta}))\approx\nabla g^\top\boldsymbol\Sigma\,\nabla g$,
and the off-diagonal entries matter: the ratio of two losses measured on
the same test set has a correlated numerator and denominator.

The rule needs $g'(\theta)\ne0$. Where the slope vanishes, as for
$g(\theta)=(\theta-\frac12)^2$ at $\theta=\frac12$, the first-order term
disappears and $g(\hat\theta)$ fluctuates on the scale $1/n$ rather than
$1/\sqrt n$. The rule also needs $\hat\theta$ to stay where the linear
approximation holds, which fails for an accuracy near 100% (exercise
5). When no derivative is convenient at
all, we can resample instead.

## The Bootstrap

Every standard error so far came from a formula. A median, an AUC or a BLEU
score varies from test set to test set just as an accuracy does, but it has
no convenient formula. This part simulates the sampling distribution
instead, first checks the simulation where the answer is known, and then
shows where it breaks.

### The Plug-in Principle

If we knew the distribution $F$ that generated the data, we could simulate
the sampling distribution of any statistic: draw many fresh data sets from
$F$ and recompute the statistic on each. The *bootstrap*
[@Efron.1979] makes one substitution. It replaces the unknown $F$ by the
*empirical distribution* $\hat F_n$, which puts mass $1/n$ on each observed
point. Drawing $n$ points from $\hat F_n$ is the same as resampling the data
$n$ times with replacement. The procedure has three steps:

1. Draw $n$ points from the data with replacement to form a *bootstrap
   resample*; some points appear several times, others not at all.
2. Compute the statistic $\hat\theta^*$ on the resample.
3. Repeat $R$ times to obtain $\hat\theta^*_1,\ldots,\hat\theta^*_R$.

The standard deviation of the replicates estimates the standard error of
$\hat\theta$, and their $\alpha/2$ and $1-\alpha/2$ quantiles form the
*percentile interval*. the figure shows the procedure. The
bootstrap assumes that the observations are i.i.d. and that the statistic
changes smoothly when the data change a little.

![The ordinary bootstrap. From one observed sample (top left), draw many resamples of the same size with replacement (bottom left; a tick's height counts how many copies of that observation were drawn) and recompute the statistic, here the median. The replicate histogram (right) estimates the sampling distribution around the observed estimate $\hat\theta$ (dashed line); its standard deviation estimates the standard error, and its central quantiles give a percentile interval. The histogram is jagged because a resample's median can take only a few of the observed values; this mild non-smoothness becomes a failure for the maximum.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prob-bootstrap.svg)

### Bootstrap Standard Errors

Before trusting the bootstrap where no formula exists, we check it where one
does. Our test set has 912 correct and 88 incorrect answers. Resampling its
1,000 indicators and recomputing the accuracy should reproduce the standard
error $0.0090$.

In [ ]:
correct = np.r_[np.ones(912), np.zeros(88)]          # the opening's test set
idx = rng.integers(0, len(correct), (5000, len(correct)))  # 5,000 resamples
boot = correct[idx].mean(axis=1)                     # accuracy of each resample
print(f'bootstrap se {boot.std(ddof=1):.4f}, '
      f'formula {np.sqrt(0.912 * 0.088 / 1000):.4f}')

The two agree to within a few percent. Now a statistic without a
convenient formula: the median of 200 draws from the exponential
distribution with mean 1, a skewed distribution whose median
$\log2\approx0.69$ lies below its mean. A formula for the median's
standard error does exist, $1/(2f(m)\sqrt n)$ with $f(m)$ the density at
the median $m$ (exercise 9), but it asks
for a density that in practice we do not know. Here we do know it,
$f(m)=e^{-\log2}=\frac12$, so the bootstrap can be held to a prediction:
$1/\sqrt{200}\approx0.071$.

In [ ]:
data = rng.exponential(1.0, 200)    # its median's se formula needs the unknown density
theta_hat = np.median(data)
idx = rng.integers(0, len(data), (10000, len(data)))  # resample with replacement
boot = np.median(data[idx], axis=1)                   # one median per resample
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f'median {theta_hat:.3f}, bootstrap se {boot.std(ddof=1):.3f}, '
      f'95% percentile interval ({lo:.3f}, {hi:.3f})')

The bootstrap never asked for the density, and its standard error is typically
near $0.07$. The printed median scatters around $\log2$, usually by less than
$0.15$, and the bootstrap standard error itself moves between about $0.05$ and
$0.11$ from one data set to the next. A resample's median can only be one of
the few observed values near the sample median, so the bootstrap distribution
is lumpy and depends on the random spacing of those values. The percentile
interval is correspondingly rough, and in some runs it only just contains
$\log2$. The median is still regular enough for the bootstrap, because the
density at the median is positive.

### Failures of the Bootstrap

The bootstrap fails when the statistic does not change smoothly with the
data. The sample maximum is the standard example. Let $x_1,\ldots,x_n$ be
i.i.d. uniform on $[0,\theta]$ and $\hat\theta=\max_i x_i$. A resample has the
same maximum as the data unless it misses the largest point in all $n$
draws, so

$$
P(\hat\theta^*=\hat\theta)=1-\Bigl(1-\frac1n\Bigr)^n \;\to\; 1-\frac1e\approx0.63 .
$$

The bootstrap distribution therefore puts 63% of its mass on a single value,
while the true sampling distribution of $\hat\theta$ is continuous. Since no
resample's maximum exceeds $\hat\theta<\theta$, the percentile interval
never contains $\theta$. Exercise 6
measures whether reflecting the interval about the estimate repairs it.

The bootstrap also fails when it resamples units that are not independent, and
the questions of an evaluation set often are not. A reading-comprehension
benchmark may hold $G$ passages with $m$ questions each, and a model that
misreads a passage tends to miss several of its questions. If the correctness
indicators of two questions on the same passage have correlation $\rho$, and
different passages are independent, the accuracy has variance

$$
\textrm{Var}(\hat\theta)=\frac{\theta(1-\theta)}{Gm}\bigl(1+(m-1)\rho\bigr) ,
$$

larger by the factor $1+(m-1)\rho$ than for $Gm$ independent questions. A
bootstrap that resamples individual questions assumes independence, so it
understates the standard error by the factor $\sqrt{1+(m-1)\rho}$, which is
$\sqrt{2.9}\approx1.7$ at $m=20$ and a modest $\rho=0.1$
[@Miller.2024]. Resampling whole passages restores the right answer
(exercise 12).

Two questions remain open. First, nothing here said which estimator to use, or
how precisely the data can pin down a parameter at all;
that section answers both with the likelihood and its
curvature. Second, every guarantee here holds for one estimator or one test
fixed before the data were seen. A model chosen because it scored best on a
validation set is not fixed in advance, since choosing the best of many
configurations is multiple testing in another form.
that section asks how large a sample must be for a
guarantee to hold for every model in a class at once.

## Summary

An estimate computed from random data is itself random, and its sampling
distribution over the data sets we did not draw decides how far it can be
trusted. Its mean squared error is the squared bias plus the variance, so a
biased estimator such as the shrunk mean can beat the unbiased sample mean.
A Monte Carlo estimate of an expectation is a sample mean of computed
samples, so its root-mean-square error is $\sigma/\sqrt n$ in every
dimension, where the error of a grid worsens with each dimension added.
A test rejects when the observed difference would be rare under the null
hypothesis; because a $p$-value is uniform under the null, it errs at rate
$\alpha$, and its power is set by the standardized effect times $\sqrt n$,
which pairing two models on the same questions raises. A confidence
interval is the set of parameter values that a test keeps. The bootstrap
simulates the sampling distribution by resampling the data; it works for
smooth statistics of independent observations and fails for the maximum and
for clustered test sets.

## Exercises

1. [code] **Shrinking the sample mean.** The text found that the shrunk mean
   $c\,\bar x$ of $x_1,\ldots,x_n$ i.i.d. $\mathcal N(\theta,\sigma^2)$ has
   the smallest MSE at $c^\star=\theta^2/(\theta^2+\sigma^2/n)$.
    1. Extend the text's simulation of the sample mean, the shrunk mean
       and the median to a grid of factors $c\in[0,1]$ at
       $\theta=1$, $\sigma=4$ and $n=30$. Plot the squared bias, the
       variance and the MSE against $c$ with `d2l.plot`, and compare the
       empirical minimizer with $c^\star$.
    1. Shrink toward a point $a$ instead of $0$, with the estimator
       $a+c(\bar x-a)$. Compute its MSE and its optimal factor. Which $a$
       would you choose, and what does the answer say about the choice of
       $0$ in the text?
    1. $c^\star$ is not an estimator, because it contains $\theta$. Plug in
       $\bar x$ for $\theta$ and $s^2$ for $\sigma^2$, giving
       $\hat c=\bar x^2/(\bar x^2+s^2/n)$, and measure the MSE of
       $\hat c\,\bar x$ by simulation at $\theta\in\{0,1,3,10\}$. Where does
       it beat $\bar x$, and where does it lose? Exercise
       13 shows that in three or more
       dimensions a similar substitution wins for every $\theta$.

   *Hint for (c):* at $\theta=0$ the plug-in factor is itself noisy; look at
   its distribution.

1. **Reading a $p$-value.** A $p$-value is computed under $H_0$, so by
   itself it does not give the probability that $H_0$ is true.
    1. A two-sided $z$-test of $H_0\!:\mu=0$ on $n=20$ observations reports
       $p=0.5$. Compute the power of this test at $\alpha=0.05$ against the
       standardized effect $\delta=0.1$. Why is $p=0.5$ then no evidence that
       $H_0$ is true?
    1. In a research program, $90\%$ of the tested ideas have no effect. Each
       idea is tested once at $\alpha=0.05$, with power $0.8$ against its
       true effect. What fraction of the significant results are false
       positives? Recompute the fraction for power $0.2$. This calculation
       is the core of the argument of @Ioannidis.2005 . Which of its
       ingredients does a $p$-value not contain?

1. [code] **Multiple comparisons.** A hyperparameter sweep compares many
   configurations with one baseline, so it runs many tests at once.
    1. Twenty models, none of them better than the baseline, are each
       compared with it by an independent test at $\alpha=0.05$. Compute the
       probability that at least one comparison is significant, and the same
       probability at the Bonferroni threshold $\alpha/20$. Show that the
       Bonferroni correction holds the probability of at least one false
       positive at or below $\alpha$ even when the tests are dependent.
    1. The Benjamini–Hochberg procedure [@Benjamini.Hochberg.1995]
       sorts the $m$ $p$-values, $p_{(1)}\le\cdots\le p_{(m)}$, finds the
       largest $k$ with $p_{(k)}\le k\alpha/m$, and rejects the $k$
       hypotheses with the smallest $p$-values. Simulate a sweep of $m=200$
       configurations, of which $m_0=180$ have no effect. Draw the
       $z$-statistics from $\mathcal N(0,1)$ for these and from
       $\mathcal N(3,1)$ for the other $20$, and compute the two-sided
       $p$-values `2 * norm.sf(np.abs(z))`. Over $1{,}000$ repetitions,
       report the average numbers of true and false discoveries and the
       average false-discovery proportion (the fraction of rejections that
       are false, or $0$ without rejections) for the uncorrected threshold
       $0.05$, for Bonferroni and for Benjamini–Hochberg. For independent
       tests, Benjamini–Hochberg guarantees an expected false-discovery
       proportion of at most $\alpha m_0/m$. Does the simulation agree?

1. [code] **Coverage at small $n$.** The interval
   the equation uses the Gaussian quantile $1.96$ in
   place of the $t$-quantile, and it assumes approximately Gaussian sample
   means.
    1. Run the coverage simulation of the text (1,000 intervals from
       standard normal data sets) with
       $n\in\{2,5,10,30,100\}$ and $100{,}000$ trials, and record the
       coverage of the interval. Then replace $1.96$ by the $97.5\%$
       quantile of the $t$-distribution with $n-1$ degrees of freedom, which
       is $12.71$, $2.776$, $2.262$, $2.045$ and $1.984$ for these $n$, and
       record the coverage again.
    1. Set $n=2$ and replace $1.96$ by $1.0$, the $75\%$ quantile of the
       $t$-distribution with one degree of freedom, so that the interval has
       $50\%$ coverage. Compute the coverage separately for the intervals
       shorter and longer than the median width. Does the difference
       contradict the equation? Would you read a short interval
       as a sign of a precise estimate?
    1. Draw the data from the exponential distribution with mean $1$ at
       $n=10$, keeping the $t$-quantile $2.262$. Measure the coverage, and
       count separately the intervals that lie entirely below and entirely
       above the true mean. Explain the asymmetry from the shape of the
       distribution.

1. [code] **Delta method near a boundary.** By the equation,
   an accuracy $\hat\theta=0.90$ measured on $n=1{,}000$ test examples has a
   log-odds $\log\frac{\hat\theta}{1-\hat\theta}$ with standard error about
   $0.105$.
    1. Simulate $100{,}000$ test sets by drawing the number of correct
       predictions from the binomial distribution with $n=1{,}000$ and
       $\theta=0.9$. Compare the standard deviation of the log-odds with the
       delta-method value.
    1. Repeat with $\theta=0.999$. Estimate the fraction of test sets with
       $\hat\theta=1$, compare it with its exact value, and state the
       log-odds of such a test set.
    1. Which assumption behind the first-order expansion fails at
       $\theta=0.999$? Compute the expected number of incorrect answers,
       $n(1-\theta)$, for both values of $\theta$, and explain why this
       count, rather than $n$ alone, decides whether the delta method and
       the Wald interval can be trusted.
    1. A benchmark has 30 questions, and a model answers 29 of them
       correctly. Compute the Wald interval
       $\hat\theta\pm1.96\sqrt{\hat\theta(1-\hat\theta)/30}$ and note where
       it ends. Then invert a test: for each $\theta_0$ on a fine grid of
       $[0.5,1]$, compute the exact two-sided $p$-value of $H_0\!:\theta=\theta_0$,
       $\min\{1,\,2\min\{P_{\theta_0}(K\le29),\,P_{\theta_0}(K\ge29)\}\}$ with
       $K\sim\mathrm{Binomial}(30,\theta_0)$, and collect the $\theta_0$ that
       it keeps at $\alpha=0.05$. Compare the two intervals, and their
       coverage at $\theta=0.95$ over $100{,}000$ simulated benchmarks.

1. [code] **Bootstrap of the maximum.** Let $x_1,\ldots,x_n$ be i.i.d.
   $U(0,\theta)$, let $\hat\theta=\max_i x_i$, and let $\hat\theta^*$ be the
   maximum of a bootstrap resample. The text showed that
   $P(\hat\theta^*=\hat\theta)\to1-1/e$.
    1. Show that $n(\theta-\hat\theta)/\theta$ converges in distribution to
       the exponential distribution with mean 1, whereas
       $n(\hat\theta-\hat\theta^*)/\hat\theta$, conditionally on the data,
       has an atom of probability about $0.63$ at $0$. Which part of the
       true sampling distribution does the bootstrap fail to reproduce?
    1. The basic bootstrap interval reflects the percentile quantiles
       $q_{\alpha/2}$ and $q_{1-\alpha/2}$ of the replicates about the
       estimate, $[2\hat\theta-q_{1-\alpha/2},\,2\hat\theta-q_{\alpha/2}]$.
       Adapt the text's bootstrap of the median to the maximum, with
       $n=100$ and $\theta=1$, and
       estimate the coverage of the basic $95\%$ interval over $1{,}000$
       data sets with $R=2{,}000$ resamples each.
    1. For one data set, plot the histogram of $n(\hat\theta-\hat\theta^*)$
       next to the histogram of $n(\theta-\hat\theta)$ over fresh data sets,
       and use the two pictures to explain the coverage you measured.

   *Adapted from Larry Wasserman,
   [All of Statistics](https://link.springer.com/book/10.1007/978-0-387-21736-9),
   Chapter 8, Exercise 7.*

1. [code] **Choosing the variance denominator.** ● Let
   $S=\sum_i(x_i-\bar x)^2$ and consider the estimators $S/c$ of $\sigma^2$.
   The plug-in variance $s_0^2$ and the unbiased estimator $s^2$ of
   the equation are the cases $c=n$ and $c=n-1$. For
   Gaussian data, $S/\sigma^2$ has the $\chi^2$ distribution with $n-1$
   degrees of freedom, so $\textrm{Var}(S)=2(n-1)\sigma^4$.
    1. Show that for Gaussian data the MSE of $S/c$ is minimized at $c=n+1$.
       Rank $c=n-1$, $n$ and $n+1$ by their MSE at $n=3$ and $\sigma^2=4$.
    1. Estimate the three MSEs by simulation, over $100{,}000$ Gaussian
       data sets with $n=3$ and $\sigma^2=4$, and compare them with your
       answer to (a).
    1. For non-Gaussian i.i.d. data with excess kurtosis
       $\kappa=\mathbb{E}[(x-\mu)^4]/\sigma^4-3$, the variance of $S$ is
       $$\textrm{Var}(S)=(n-1)^2\sigma^4\left(\frac{\kappa}{n}+\frac{2}{n-1}\right).$$
       Find the MSE-optimal $c$ as a function of $n$ and $\kappa$. Evaluate
       it for Laplace data, which have $\kappa=3$, at $n=3$ and $n=10$, and
       check the case $n=3$ by simulation.
    1. The optimal shrinkage factor $c^\star$ of the text depends on the
       unknown $\theta$, whereas for Gaussian data the optimal denominator
       depends on $n$ alone. Why? What does (c) imply for the use of an
       MSE-optimal denominator when the distribution of the data is
       unknown?

1. [code] **Paired comparisons.** ● Two models evaluated on the same test
   examples or the same seeds share part of their noise, and a paired test
   removes it.
    1. In the text the two models disagree on 41 questions, 25 to 16. Under
       $H_0$ the number of questions that only B gets right is
       $\mathrm{Binomial}(41,\tfrac12)$. Compute the exact two-sided
       $p$-value and compare it with the Monte Carlo value of the
       sign-flip cell in that section. Then fix
       the net gap at 9 and vary the number of disagreements: compute $p$
       for 9 to 0, 17 to 8, 25 to 16 and 55 to 46. What decides the
       verdict, the gap or the number of disagreements?
    1. Model the per-seed accuracies with a seed effect that both models
       share: for seed $s$, the accuracies are $0.850+u_s+e_{A,s}$ and
       $0.858+u_s+e_{B,s}$, where $u_s\sim\mathcal N(0,0.01^2)$ and
       $e_{A,s},e_{B,s}\sim\mathcal N(0,0.003^2)$ are independent. Implement
       the sign-flip test on the per-seed differences as an exact test:
       with $k$ seeds, enumerate all $2^k$ sign patterns instead of drawing
       random ones. Over $1{,}000$ experiments with $10$ seeds, compare its
       power with that of the unpaired permutation test `perm_p`.
    1. Repeat with $5$ seeds. What is the smallest two-sided $p$-value that
       the exact sign-flip test can return, and what does it imply at
       $\alpha=0.05$? A Monte Carlo version that draws $999$ random sign
       patterns, as `perm_p` draws random shuffles, rejects in a few
       percent of these experiments. Why?

1. **Standard error of the median.** ● Let $x_1,\ldots,x_n$ be i.i.d. with
   c.d.f. $F$, median $m$, and density $f(m)>0$ at the median. The sample
   median $\hat m$ is at most $m+t$ exactly when at least half of the
   observations are at most $m+t$.
    1. The empirical c.d.f. $\hat F_n(m+t)$ has mean $F(m+t)$ and variance
       $F(m+t)\bigl(1-F(m+t)\bigr)/n$. Use the central limit theorem to show
       that $P(\hat m\le m+t)\approx\Phi\bigl(2\sqrt n\,f(m)\,t\bigr)$ for
       small $t$, and conclude that
       $\textrm{se}(\hat m)\approx 1/\bigl(2f(m)\sqrt n\bigr)$. Relate
       the factor $1/f(m)$ to the slope $g'$ in the equation.
    1. Evaluate the approximation for the exponential distribution with
       mean $1$ at $n=200$, and compare it with the bootstrap standard error
       that your run of the text's bootstrap of the median prints. Rerun
       the cell several times: the printed value moves between about $0.05$
       and $0.11$. Why does the bootstrap estimate of a median's standard error
       vary this much from data set to data set, when the bootstrap estimate
       of a mean's standard error does not?
    1. Compute the ratio $\textrm{Var}(\hat m)/\textrm{Var}(\bar x)$ for
       large $n$ for Gaussian data and for Laplace data. Which estimator of
       the location would you choose for heavy-tailed data?

   *Hint for (b):* a resample's median is one of a few observed values near
   $\hat m$, and the spacing of those values is random.

1. [code] **Grids against Monte Carlo.** ●
   that section compares exponents: with
   $N$ evaluations in $d$ dimensions, a second-order grid has error of order
   $N^{-2/d}$ and Monte Carlo of order $N^{-1/2}$. Test this comparison on
   $f(\mathbf x)=\exp(-\|\mathbf x\|^2)$ over $[0,1]^d$, whose integral is
   $\bigl(\tfrac{\sqrt\pi}{2}\operatorname{erf}(1)\bigr)^d$ by the product
   rule the equation.
    1. Implement the midpoint rule on the full grid of $m^d$ cubes and the
       Monte Carlo average the equation over $N=m^d$ uniform
       samples. For $d\in\{1,2,4,5,6\}$ and $N$ up to about $10^6$, plot the
       relative error against $N$ on log-log axes, using for Monte Carlo the
       root mean squared error over $20$ repetitions. Do the slopes match
       the predicted exponents?
    1. At $d=4$ the exponents tie. Which method is more accurate there?
       What sets the constant in front of each rate, and how does each
       constant change with $d$?
    1. Both errors can be predicted. The midpoint sum of $f$ is a product of
       one-dimensional midpoint sums. The root mean squared error of Monte
       Carlo is $\mathrm{sd}\bigl(f(\mathbf U)\bigr)/\sqrt N$ by
       the equation, for $\mathbf U$ uniform on $[0,1]^d$, and
       the product rule factors $\mathbb{E}\bigl[f(\mathbf U)^2\bigr]$ into
       one-dimensional integrals. Compute both predicted relative errors for
       $d\in\{4,5,6\}$ at $N\approx 10^5$ and compare them with your
       measurements. At which $d$ do the two methods break even? How reliably
       do $20$ repetitions decide the comparison at that $d$?

1. [code] **Monte Carlo under heavy tails.** ● The Monte Carlo average
   the equation converges by the law of large numbers when
   the mean is finite, and its $1/\sqrt n$ rate rests on the finite variance
   in the equation. Use the Pareto and Cauchy densities of
   exercise 3, and write
   $\bar X_n$ for the running mean of the first $n$ samples.
    1. Show that $X=U^{-1/\alpha}$, with $U$ uniform on $(0,1)$, has the
       Pareto density with tail index $\alpha$, by computing $P(X>x)$. Find
       the analogous transformation of $U$ for the Cauchy density.
    1. For $\alpha=3$ and $\alpha=1.5$, draw $200$ independent sequences of
       $10^5$ samples each. Plot the median over sequences of
       $|\bar X_n-\mathbb{E}[X]|$ against $n$ on log-log axes, and fit the
       slopes. Why is the median a better summary here than the root mean
       squared error?
    1. Show that the largest of $n$ Pareto samples exceeds $n^{1/\alpha}$
       with probability approaching $1-e^{-1}$ as $n\to\infty$. A single
       sample of that size shifts $\bar X_n$ by about $n^{1/\alpha-1}$. Use
       this to predict the slope for $1<\alpha<2$. Why does the same
       argument not set the slope for $\alpha=3$?
    1. The Pareto density with $\alpha=1$ and the Cauchy density have no
       finite mean, so the error of (b) is undefined for them. Draw $200$
       sequences of $10^5$ samples from each, and plot the 25th, 50th and
       75th percentiles of $\bar X_n$ over sequences against $n$. What does
       the running mean do in each case?

1. [code] **Clustered test sets.** ● Evaluation sets often consist of
   clusters of related items, such as several questions about one passage,
   and the i.i.d. bootstrap then understates the standard error
   [@Miller.2024]. A test set has $G$ clusters of $m$ items each, its
   expected accuracy is $\theta$, the correctness indicators of two items in
   the same cluster have correlation $\rho$, and items in different clusters
   are independent.
    1. Show that the average of $m$ variables with common variance $v$ and
       pairwise correlation $\rho$ has variance $v\bigl(1+(m-1)\rho\bigr)/m$,
       and conclude that the accuracy $\hat\theta$ on the test set has
       variance
       $$\textrm{Var}(\hat\theta)=\frac{\theta(1-\theta)}{Gm}\bigl(1+(m-1)\rho\bigr).$$
    1. Simulate $G=100$ and $m=20$, drawing a cluster accuracy
       $\theta_g\sim\mathrm{Beta}(6.3,2.7)$ for each cluster and
       conditionally independent items given $\theta_g$. These choices give
       $\theta=0.7$ and $\rho=1/(6.3+2.7+1)=0.1$. Over $300$ test sets,
       compare the standard deviation of $\hat\theta$ with the average
       i.i.d. bootstrap standard error, which resamples items as in the
       text's bootstrap of the accuracy, and with the average cluster
       bootstrap standard error, which resamples whole clusters, using
       $R=1{,}000$ resamples for each. By which factor does the i.i.d.
       bootstrap understate the standard error? Compare the factor with
       the prediction of (a).

1. [code] **Stein's paradox.** ●● Observe $\mathbf x\sim\mathcal
   N(\boldsymbol\theta,\mathbf I_d)$ and measure an estimator by its risk
   $\mathbb{E}\|\hat{\boldsymbol\theta}-\boldsymbol\theta\|^2$, the MSE with
   squares replaced by squared norms. The estimator $\mathbf x$ has risk $d$
   for every $\boldsymbol\theta$. @James.Stein.1961 showed that for
   $d\ge3$ the estimator
   $$\hat{\boldsymbol\theta}_{\textrm{JS}}=\left(1-\frac{d-2}{\|\mathbf x\|^2}\right)\mathbf x$$
   has smaller risk for every $\boldsymbol\theta$.
    1. Prove Stein's lemma [@Stein.1981]: if $z\sim\mathcal N(\theta,1)$
       and $h$ is differentiable with $\mathbb{E}|h'(z)|<\infty$, then
       $\mathbb{E}[(z-\theta)h(z)]=\mathbb{E}[h'(z)]$. Integrate by parts.
    1. For an estimator $\mathbf x+\mathbf g(\mathbf x)$ with $\mathbf g$
       weakly differentiable (here: differentiable away from the origin,
       with $\mathbb{E}\|\mathbf g(\mathbf x)\|^2<\infty$), show that its
       risk is
       $$d+\mathbb{E}\|\mathbf g(\mathbf x)\|^2+2\sum_{i=1}^d\mathbb{E}\left[\frac{\partial g_i}{\partial x_i}(\mathbf x)\right].$$
       Use this identity to show that
       $\hat{\boldsymbol\theta}_{\textrm{JS}}$ has risk
       $d-(d-2)^2\,\mathbb{E}\bigl[1/\|\mathbf x\|^2\bigr]$. Where does the
       argument need $d\ge3$?
    1. Simulate the risks of $\mathbf x$, of
       $\hat{\boldsymbol\theta}_{\textrm{JS}}$, and of the positive-part
       estimator $\max\bigl(0,1-(d-2)/\|\mathbf x\|^2\bigr)\mathbf x$ for
       $d=10$ and $\|\boldsymbol\theta\|\in\{0,1,2,5,10,20\}$, and plot them
       against $\|\boldsymbol\theta\|$.
    1. Write the optimal factor $c^\star$ for $c\,\mathbf x$ in $d$
       dimensions, the analogue of the text's $c^\star$. Which function of
       the data replaces the unknown quantity in
       $\hat{\boldsymbol\theta}_{\textrm{JS}}$, and why does this substitute
       become accurate as $d$ grows? Compare with exercise
       1c.

1. [extended] **Peeking at a running test.** ●● An A/B test, or an
   evaluation during training, is often watched while the data arrive and
   stopped as soon as the result looks significant
   [@Johari.Koomen.Pekelis.ea.2017]. Let $x_1,x_2,\ldots$ be i.i.d.
   $\mathcal N(\mu,1)$, let $H_0\!:\mu=0$, and let $S_n=\sum_{i\le n}x_i$.
   The $z$-test at time $n$ rejects when $|S_n|/\sqrt n\ge1.96$.
    1. Simulate $4{,}000$ runs of length $N=10{,}000$ under $H_0$. Estimate
       the probability that at least one of $K$ equally spaced looks rejects,
       for $K\in\{1,2,5,10,100,1000,10000\}$. The law of the iterated
       logarithm implies that this probability tends to one as monitoring
       continues indefinitely. How fast does it grow in your table?
    1. Put the prior $\mathcal N(0,\tau^2)$ on $\mu$ and define the mixture
       likelihood ratio
       $$\Lambda_n=\int\prod_{i=1}^n\frac{p(x_i\mid\mu)}{p(x_i\mid0)}\,\mathcal N(\mu;0,\tau^2)\,d\mu.$$
       Show that
       $\Lambda_n=(1+n\tau^2)^{-1/2}\exp\bigl(\tau^2S_n^2/(2(1+n\tau^2))\bigr)$,
       and that under $H_0$ it is a martingale:
       $\mathbb{E}[\Lambda_{n+1}\mid x_1,\ldots,x_n]=\Lambda_n$.
    1. Ville's inequality, which we take on faith, states that a nonnegative
       martingale with $\Lambda_0=1$ satisfies
       $P(\sup_n\Lambda_n\ge1/\alpha)\le\alpha$. Conclude that rejecting at
       the first $n$ with $\Lambda_n\ge1/\alpha$ has type I error at most
       $\alpha$, however often the data are inspected. Derive the rejection
       boundary for $|S_n|$, and compare its growth in $n$ with that of the
       fixed boundary $1.96\sqrt n$.
    1. Check the type I error of the sequential test by simulation for
       $\tau^2\in\{0.01,0.1,1\}$. Then set $\mu=0.2$, and take the
       fixed-sample $z$-test whose $n$ follows from
       the equation for power $0.8$ as the reference.
       For each of the three values of $\tau^2$, report the power of the
       sequential test by that $n$, its median stopping time, and its power
       by $N$. How does the choice of $\tau^2$ affect these numbers? What
       does a test that is valid at every look give up, and what does it gain?

[Discussions](https://d2l.discourse.group/t/1103)